#   libraries

In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn. feature_extraction. text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GridSearchCV
import pickle

#   Data

In [ ]:
true = pd.read_csv(r'F:\Documents\ML & AI\Projects\data\News\True.csv')
fake = pd.read_csv(r'F:\Documents\ML & AI\Projects\data\News\Fake.csv')

In [ ]:
true.head()

In [ ]:
fake.head()

In [ ]:
true['label'] = 1
fake['label'] = 0


In [ ]:
true.head()

In [ ]:
fake.head()

In [ ]:
news = pd.concat([fake , true] , axis=0)

In [ ]:
news.head()

In [ ]:
news.tail()

In [ ]:
news.isnull().sum()

In [ ]:
news = news.drop(['title' , 'subject' ,'date'] , axis = 1)

In [ ]:
news.head()

In [ ]:
news = news.sample(frac=1).reset_index(drop=True)

*   generalize the text formate

In [ ]:
def wordopt(text):
    # 1. تحويل النص كله لحروف صغيرة
    text = text.lower()
    
    # 2. حذف كلمة reuters تحديداً لاختبار قوة الموديل
    # استخدمنا \b قبل وبعد الكلمة عشان نتأكد إنه بيمسح الكلمة المستقلة بس
    text = re.sub(r'\breuters\b', '', text)
    
    # 3. حذف الروابط (بعد إصلاح مسافات التعبير النمطي)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 4. حذف وسوم HTML (بعد إصلاح المسافات)
    text = re.sub(r'<.*?>', '', text)
    
    # 5. حذف علامات الترقيم
    text = re.sub(r'[^\w\s]', '', text)
    
    # 6. حذف الأرقام
    text = re.sub(r'\d', '', text)
    
    # 7. استبدال النزول لسطر جديد بمسافة فاضية (عشان الكلمات متلزقش في بعض)
    text = re.sub(r'\n', ' ', text)
    
    return text

In [ ]:
news['text'] = news['text'].apply(wordopt)

#   Splitting The Data

In [ ]:
X = news['text']
Y = news ['label']

x_train , x_test , y_train , y_test = train_test_split(X , Y , test_size=0.3 , random_state=42)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

*   Turning The Data Into Numeric

In [ ]:
vectorization = TfidfVectorizer()

In [ ]:
x_trainV = vectorization.fit_transform(x_train)

In [ ]:
x_testV = vectorization.transform(x_test)

In [ ]:
x_trainV

In [ ]:
x_testV

#   Making The Model Using Logistic Regression

In [ ]:
LRmodel = LogisticRegression()

LRmodel.fit(x_trainV , y_train)


In [ ]:
LRprediction = LRmodel.predict(x_testV)

In [ ]:
LRmodel.score(x_testV , y_test) * 100

In [ ]:
print(classification_report(y_test, LRprediction ))

#   Making The Model Using Decision Tree

In [ ]:
DTmodel = DecisionTreeClassifier(max_depth=5)
DTmodel.fit(x_trainV,y_train)

In [ ]:
DTprediction = DTmodel.predict(x_testV)

In [ ]:
DTmodel.score(x_testV , y_test) * 100 

In [ ]:
print(classification_report(y_test, DTprediction ))           # This model is Overfitting

In [ ]:

# 1. تحديد قائمة الإعدادات اللي عايزين الأداة تجربها
param_grid = {
    'max_depth': [5, 10, 20, 50, None],        # عمق الشجرة
    'min_samples_split': [2, 10, 20],          # أقل عدد من البيانات عشان الشجرة تتفرع
    'criterion': ['gini', 'entropy']           # طريقة حساب التقسيم
}

# 2. تجهيز الموديل والأداة
dt_base = DecisionTreeClassifier()

# cv=3 معناها بيقسم البيانات 3 أجزاء للاختبار عشان يتأكد من الدقة
# n_jobs=-1 بتخلي الأداة تستخدم كل أنوية المعالج (CPU) عشان تسرع البحث
grid_search = GridSearchCV(estimator=dt_base, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# 3. بدء عملية البحث والتدريب (ممكن تاخد دقيقة أو دقيقتين)
print("جاري البحث عن أفضل إعدادات... انتظر قليلاً")
grid_search.fit(x_trainV, y_train)

# 4. طباعة أفضل إعدادات الأداة لقتها
print("أفضل إعدادات تم العثور عليها:")
print(grid_search.best_params_)

# 5. استخراج أفضل موديل وعمل توقع بيه
tuned_DTmodel = grid_search.best_estimator_
tuned_DTprediction = tuned_DTmodel.predict(x_testV)

# 6. طباعة الدقة الجديدة
tuned_dt_score = tuned_DTmodel.score(x_testV, y_test) * 100
print(f"\n Model accuracy after tunning {tuned_dt_score:.2f}%")

#   Making The Model Using Random Forest


In [ ]:
RFmodel = RandomForestClassifier()
RFmodel.fit(x_trainV,y_train)

In [ ]:
RFprediction = RFmodel.predict(x_testV)

In [ ]:
RFmodel.score(x_testV , y_test) * 100 

In [ ]:
print(classification_report(y_test, RFprediction ))

#   Making The Model Using Gradient Boosting Classifier.

In [ ]:
GBmodel = GradientBoostingClassifier()
GBmodel.fit(x_trainV,y_train)

In [ ]:
GBprediction = GBmodel.predict(x_testV)

In [ ]:
GBmodel.score(x_testV , y_test) * 100

In [ ]:
print(classification_report(y_test, GBprediction ))

#   Improves

In [ ]:
def output_label(n):
    if n == 0:
        return "It is a Fake News"
    elif n == 1:
        return "It is a Genuine News"

In [ ]:
def manual_testing(news):
    testing_news = {"text": [news]} # Corrected syntax for defining dictionary

    new_def_test = pd.DataFrame(testing_news)

    new_def_test["text"] = new_def_test["text"].apply(wordopt)

    new_x_test = new_def_test["text"]

    new_xv_test = vectorization.transform(new_x_test) # Assuming 'vectorization' is your vectorizer object

    pred_lr = LRmodel.predict(new_xv_test)

    # pred_dtc = dtc.predict(new_xv_test)

    pred_gbc = GBmodel.predict(new_xv_test)

    pred_rfc = RFmodel.predict(new_xv_test)

    return "\n\nLR Prediction: {} \nGBC Prediction: {} \nRFC Prediction: {}".format(

        output_label(pred_lr[0]), output_label(pred_gbc[0]), output_label(pred_rfc[0]))

In [ ]:
# news_article = str(input())

In [ ]:
# manual_testing(news_article)

#   Comparing & Visualising

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. تجميع نسب الدقة لكل الموديلات (بما فيها الـ Decision Tree القديم والجديد)
lr_score = LRmodel.score(x_testV, y_test) * 100
dt_score = DTmodel.score(x_testV, y_test) * 100               # الشجرة القديمة (100%)
tuned_dt_score = tuned_DTmodel.score(x_testV, y_test) * 100   # الشجرة المحسنة الواقعية
rf_score = RFmodel.score(x_testV, y_test) * 100
gb_score = GBmodel.score(x_testV, y_test) * 100

# 2. إنشاء جدول للمقارنة وترتيبه
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree (Original)', 'Decision Tree (Tuned)', 'Random Forest', 'Gradient Boosting'],
    'Accuracy (%)': [lr_score, dt_score, tuned_dt_score, rf_score, gb_score]
})

comparison_df = comparison_df.sort_values(by='Accuracy (%)', ascending=False).reset_index(drop=True)

# طباعة الجدول
print("جدول مقارنة أداء الموديلات (قبل وبعد التحسين):")
print("-" * 55)
print(comparison_df)
print("\n")

# 3. رسم بياني (Bar Chart) للمقارنة
plt.figure(figsize=(12, 6)) # كبرنا العرض عشان يستوعب 5 أعمدة براحة
bars = plt.bar(comparison_df['Model'], comparison_df['Accuracy (%)'], 
               color=['#4CAF50', '#2196F3', '#03A9F4', '#FFC107', '#F44336'])

# إضافة الأرقام فوق كل عمود
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.3, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Fake News Detection Models Comparison', fontsize=15, fontweight='bold')
plt.xlabel('Machine Learning Models', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(80, 105) 
plt.xticks(rotation=15) # لفة بسيطة للأسماء عشان تكون مقروءة بشكل أفضل
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#   Model Exporting 

In [ ]:
import pickle

# 1. حفظ أداة الـ Vectorizer (أهم خطوة لأن الموديلات مش بتفهم غير الأرقام اللي بتطلع منها)
with open('vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorization, file)

# 2. حفظ موديل الانحدار اللوجستي
with open('LR_model.pkl', 'wb') as file:
    pickle.dump(LRmodel, file)

# 3. حفظ موديل شجرة القرار (الأصلي)
with open('DT_model_original.pkl', 'wb') as file:
    pickle.dump(DTmodel, file)

# 4. حفظ موديل شجرة القرار (المحسن)
with open('tuned_DT_model.pkl', 'wb') as file:
    pickle.dump(tuned_DTmodel, file)

# 5. حفظ موديل الغابة العشوائية (Random Forest)
with open('RF_model.pkl', 'wb') as file:
    pickle.dump(RFmodel, file)

# 6. حفظ موديل تعزيز التدرج (Gradient Boosting)
with open('GB_model.pkl', 'wb') as file:
    pickle.dump(GBmodel, file)

print("تم حفظ جميع الموديلات (5 موديلات) وأداة الـ Vectorizer بنجاح!")